# Data Preprocessing
:label:`sec_pandas`

So far, we have been working with synthetic data
that arrived in ready-made tensors.
However, to apply deep learning in the wild
we must extract messy data
stored in arbitrary formats,
and preprocess it to suit our needs.
Fortunately, the *pandas* [library](https://pandas.pydata.org/)
can do much of the heavy lifting.
This section, while no substitute
for a proper *pandas* [tutorial](https://pandas.pydata.org/pandas-docs/stable/user_guide/10min.html),
will give you a crash course
on some of the most common routines.

## Reading the Dataset

Comma-separated values (CSV) files are ubiquitous
for the storing of tabular (spreadsheet-like) data.
In them, each line corresponds to one record
and consists of several (comma-separated) fields, e.g.,
"Albert Einstein,March 14 1879,Ulm,Federal polytechnic school,field of gravitational physics".
To demonstrate how to load CSV files with `pandas`,
we (**create a CSV file below**) `../data/house_tiny.csv`.
This file represents a dataset of homes,
where each row corresponds to a distinct home
and the columns correspond to the number of rooms (`NumRooms`),
the roof type (`RoofType`), and the price (`Price`).


In [ ]:
import os

os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('''NumRooms,RoofType,Price
NA,NA,127500
2,NA,106000
4,Slate,178100
NA,NA,140000''')

Now let's import `pandas` and load the dataset with `read_csv`.


In [ ]:
import pandas as pd

data = pd.read_csv(data_file)
print(data)

   NumRooms RoofType   Price
0       NaN      NaN  127500
1       2.0      NaN  106000
2       4.0    Slate  178100
3       NaN      NaN  140000


## Data Preparation

In supervised learning, we train models
to predict a designated *target* value,
given some set of *input* values.
Our first step in processing the dataset
is to separate out columns corresponding
to input versus target values.
We can select columns either by name or
via integer-location based indexing (`iloc`).

You might have noticed that `pandas` replaced
all CSV entries with value `NA`
with a special `NaN` (*not a number*) value.
This can also happen whenever an entry is empty,
e.g., "3,,,270000".
These are called *missing values*
and they are the "bed bugs" of data science,
a persistent menace that you will confront
throughout your career.
Depending upon the context,
missing values might be handled
either via *imputation* or *deletion*.
Imputation replaces missing values
with estimates of their values
while deletion simply discards
either those rows or those columns
that contain missing values.

Here are some common imputation heuristics.
[**For categorical input fields,
we can treat `NaN` as a category.**]
Since the `RoofType` column takes values `Slate` and `NaN`,
`pandas` can convert this column
into two columns `RoofType_Slate` and `RoofType_nan`.
A row whose roof type is `Slate` will set values
of `RoofType_Slate` and `RoofType_nan` to 1 and 0, respectively.
The converse holds for a row with a missing `RoofType` value.


In [ ]:
inputs, targets = data.iloc[:, 0:2], data.iloc[:, 2]
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)

   NumRooms  RoofType_Slate  RoofType_nan
0       NaN           False          True
1       2.0           False          True
2       4.0            True         False
3       NaN           False          True


For missing numerical values,
one common heuristic is to
[**replace the `NaN` entries with
the mean value of the corresponding column**].


In [ ]:
inputs = inputs.fillna(inputs.mean())
print(inputs)

   NumRooms  RoofType_Slate  RoofType_nan
0       3.0           False          True
1       2.0           False          True
2       4.0            True         False
3       3.0           False          True


## Conversion to the Tensor Format

Now that [**all the entries in `inputs` and `targets` are numerical,
we can load them into a tensor**] (recall :numref:`sec_ndarray`).


In [ ]:
import torch

X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(targets.to_numpy(dtype=float))
X, y

(tensor([[3., 0., 1.],
         [2., 0., 1.],
         [4., 1., 0.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

## Discussion

You now know how to partition data columns,
impute missing variables,
and load `pandas` data into tensors.
In :numref:`sec_kaggle_house`, you will
pick up some more data processing skills.
While this crash course kept things simple,
data processing can get hairy.
For example, rather than arriving in a single CSV file,
our dataset might be spread across multiple files
extracted from a relational database.
For instance, in an e-commerce application,
customer addresses might live in one table
and purchase data in another.
Moreover, practitioners face myriad data types
beyond categorical and numeric, for example,
text strings, images,
audio data, and point clouds.
Oftentimes, advanced tools and efficient algorithms
are required in order to prevent data processing from becoming
the biggest bottleneck in the machine learning pipeline.
These problems will arise when we get to
computer vision and natural language processing.
Finally, we must pay attention to data quality.
Real-world datasets are often plagued
by outliers, faulty measurements from sensors, and recording errors,
which must be addressed before
feeding the data into any model.
Data visualization tools such as [seaborn](https://seaborn.pydata.org/),
[Bokeh](https://docs.bokeh.org/), or [matplotlib](https://matplotlib.org/)
can help you to manually inspect the data
and develop intuitions about
the type of problems you may need to address.


## Exercises

1. Try loading datasets, e.g., Abalone from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets.php) and inspect their properties. What fraction of them has missing values? What fraction of the variables is numerical, categorical, or text?
1. Try indexing and selecting data columns by name rather than by column number. The pandas documentation on [indexing](https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html) has further details on how to do this.
1. How large a dataset do you think you could load this way? What might be the limitations? Hint: consider the time to read the data, representation, processing, and memory footprint. Try this out on your laptop. What happens if you try it out on a server?
1. How would you deal with data that has a very large number of categories? What if the category labels are all unique? Should you include the latter?
1. What alternatives to pandas can you think of? How about [loading NumPy tensors from a file](https://numpy.org/doc/stable/reference/generated/numpy.load.html)? Check out [Pillow](https://python-pillow.org/), the Python Imaging Library.


In [9]:
import os
import pandas as pd
import torch

data_file = os.path.join('..', 'content', 'sample_data', 'california_housing_train.csv')
data = pd.read_csv(data_file)

missing_fraction = data.isna().mean()
print(missing_fraction)

variable_fraction = data.dtypes.value_counts()
print(variable_fraction)

longitude             0.0
latitude              0.0
housing_median_age    0.0
total_rooms           0.0
total_bedrooms        0.0
population            0.0
households            0.0
median_income         0.0
median_house_value    0.0
dtype: float64
float64    9
Name: count, dtype: int64


In [11]:
data.loc[1:10,'longitude' : 'population']

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0
2,-114.56,33.69,17.0,720.0,174.0,333.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0
5,-114.58,33.63,29.0,1387.0,236.0,671.0
6,-114.58,33.61,25.0,2907.0,680.0,1841.0
7,-114.59,34.83,41.0,812.0,168.0,375.0
8,-114.59,33.61,34.0,4789.0,1175.0,3134.0
9,-114.60,34.83,46.0,1497.0,309.0,787.0
10,-114.60,33.62,16.0,3741.0,801.0,2434.0


As a general rule of thumb when using pandas:

Comfort Zone: Datasets up to 1 GB (typically loads and processes very quickly).
Practical Limit: Datasets up to 10% to 20% of your available RAM. For a typical laptop with 16 GB of RAM, this means a dataset of 1.5 GB to 3 GB (uncompressed CSV on disk) is the practical limit for comfortable interactive work.
Here is a breakdown of the key limitations and considerations:

1. Memory Footprint (The Biggest Bottleneck)
Memory Amplification: A CSV file on disk is stored as raw text. When pandas reads it into a DataFrame, it converts text to in-memory Python objects and data types (like float64 or object). This often causes the in-memory representation to be 2x to 10x larger than the raw CSV file size on disk.
Intermediate Copies: Many pandas operations (like filtering, grouping, or joining) do not happen in-place; they create copy DataFrames in memory. If your dataset takes up 50% of your RAM just by loading it, any subsequent operation will likely trigger an Out of Memory (OOM) error or crash your kernel.
2. Time to Read and Parse (I/O and CPU)
Parsing a CSV is a single-threaded CPU-bound process in standard pandas. Reading a multi-gigabyte text file, parsing delimiters, and inferring datatypes can take several minutes.
3. Processing & Representation
GIL (Global Interpreter Lock): Pandas is single-threaded for most operations. It does not natively distribute computations across multiple CPU cores.
Object Datatypes: If your dataset contains a lot of text columns, pandas stores them as Python object pointers, which drastically increases memory overhead and slows down processing compared to contiguous block arrays (like floats or integers).


Laptop vs. Server

On a Laptop:
Limits: Typically restricted to 8 GB–32 GB of RAM.
What happens: If you exceed the physical RAM, your operating system will start using swap memory (writing temporary RAM data to your SSD/Hard Drive). This slows down performance dramatically (often by a factor of 100x or more), making the notebook unresponsive.

On a Server:
Limits: Servers easily scale to 64 GB, 256 GB, or even Terabytes of RAM.
What happens: You can load much larger datasets (e.g., a 20 GB CSV) directly into memory without crashing. However, because standard pandas is single-threaded, processing times (like .groupby or .apply over hundreds of millions of rows) will still be very slow because it cannot leverage the high core counts of server CPUs.

Alternatives for Larger-than-Memory Datasets
If your dataset is too large for standard pandas, you can use:

1. Chunking: Read the CSV in smaller batches using pd.read_csv(file, chunksize=10000).
2. Dask / Polars: Libraries designed for parallel computing and lazy evaluation that handle datasets larger than your RAM.
3. PySpark: For distributed computing across server clusters.

1. How to deal with a very large number of categories (High-Cardinality):

Using one-hot encoding on variables with hundreds or thousands of categories (e.g., zip codes, city names) creates a massive, sparse matrix that consumes a lot of memory and can lead to overfitting (the "curse of dimensionality"). To deal with this:

a. Target Encoding (Mean Encoding): Replace each category with the average target value for that category. For example, if you are predicting house prices, replace each 'Zip Code' category with the average house price in that zip code. (Note: use smoothing or cross-validation to prevent target leakage).

b. Frequency/Count Encoding: Replace each category with its frequency or total count in the dataset. This helps the model understand how common or rare a category is.

c. Feature Hashing (The Hashing Trick): Map categories to a fixed number of numerical bins using a hash function. This controls memory usage at the cost of potential hash collisions.

d. Grouping Rare Categories: Group all infrequent categories that fall below a certain threshold (e.g., < 1% of the dataset) into a single generic category like 'Other' before encoding.

e. Entity Embeddings: For deep learning models, map categories to low-dimensional continuous vector spaces (embeddings) that are trained along with the neural network.

2. What if the category labels are all unique?

If every single row has a completely unique category label (for example: User IDs, Transaction UUIDs, Social Security Numbers, or Email Addresses), these are essentially individual row identifiers.

3. Should you include unique category labels?
No, you should exclude them from your training features.

Here is why:

a. Overfitting: Since each ID uniquely maps to exactly one target output, a machine learning model will simply memorize the mapping (e.g., ID 1234 always equals price $100,000). The model will achieve perfect accuracy on the training set but will fail completely on new data (generalization) because a new data point will have an unseen ID.

b. No Predictive Power: These identifiers do not represent generalizable patterns or relationships.

c. Memory waste: One-hot encoding $N$$N$ unique IDs for $N$$N$ rows would create an $N \times N$$N \times N$ identity matrix, which is computationally expensive and useless.

Exceptions: While you exclude them from the model training process, you keep them as metadata in your DataFrame to trace predictions back to specific records later, or use them to perform database joins.

[Discussions](https://discuss.d2l.ai/t/29)
